---
authors:
  - edesz
date: 2026-05-11
---

# Get Data

## About

In this step, we will retrieve the raw dataset and store it in a private Cloudflare R2 bucket.

:::{important} Outputs
The data will be stored as a `.xlsx` file in the root of the R2 bucket.
:::

## Python Imports

We will import `pandas` to load the raw data

In [ ]:
import os
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd
from dotenv import load_dotenv

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

## User Inputs

Below we define variables that will be used later

1. `url`
   - web URL containing the credit card customer churn data
2. `r2_key_raw_data`
   - filename in which data will be stored in R2 bucket

In [ ]:
# web url of raw dataset
url = (
    "https://raw.githubusercontent.com/azar-s91/dataset/refs/heads/master/"
    "BankChurners.csv"
)

# rename column names to include the description for numerical (continuous)
# columns
col_renamer = {
    "Months_on_book": "Months_on_book (Length of relationship with bansk[months])",
    "Total_Relationship_Count": "Total_Relationship_Count (How many products with customer) ",
    "Months_Inactive_12_mon": "Months_Inactive_12_mon (Card not used)",
    "Contacts_Count_12_mon": "Contacts_Count_12_mon (Number of contacts in 12 months)",
    "Total_Revolving_Bal": "Total_Revolving_Bal (Balance unpaid at month end)",
    "Avg_Open_To_Buy": "Avg_Open_To_Buy (Difference between the credit limit and the balance)",
    "Total_Amt_Chng_Q4_Q1": "Total_Amt_Chng_Q4_Q1(Ratio Q4/Q1)",
    "Total_Trans_Amt": "Total_Trans_Amt ( Total Transactions Value 12 months)",
    "Total_Trans_Ct": "Total_Trans_Ct (Transaction Count 12 months)",
    "Total_Ct_Chng_Q4_Q1": "Total_Ct_Chng_Q4_Q1 (Change in the transaction amount Q4/Q1)",
    "Avg_Utilization_Ratio": "Avg_Utilization_Ratio (Credit usage/Total Credit available)",
}

# name of raw data key (file) in private R2 bucket
# (note: next step 01_split_data.ipynb expects "BankChurners.xlsx")
r2_key_raw_data = "<filename-here>"

:::{warning}
[The next step](./01_split_data.ipynb) expects the `.xlsx` file to be named `BankChurners.xlsx`. So, if a filename other than `BankChurners.xlsx` is used here then it should also be changed in the next step.
:::

We will now use environment variables to define an authenticated `boto3` R2 client

In [ ]:
account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Extract

We will first load all data and rename the columns

In [ ]:
df = pd.read_csv(url).rename(columns=col_renamer)

## Transform

No data transformation is required since this is the raw data.

## Load

Finally, we'll export the data to `.xlsx` file in the R2 bucket.

In [ ]:
with BytesIO() as output:
    # use pandas.ExcelWriter to write to BytesIO buffer
    with pd.ExcelWriter(output, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=False, sheet_name="Sheet1")

    # get the binary data
    data = output.getvalue()

    # upload retrieved data to R2 bucket
    s3_client.put_object(Bucket=bucket_name, Key=r2_key_raw_data, Body=data)